# 07 — Inference & Model Export

**Project:** Ecommerce-LLM-Finetuning
**Goal of this notebook:**
1. Load the base model + LoRA adapter
2. Run an interactive chatbot loop
3. Merge LoRA weights into the base model
4. Save the merged, standalone model for deployment (Streamlit app / HF Hub)

Run this after `05_finetune_llm.ipynb` (and optionally `06_evaluation.ipynb`).


In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"


In [ ]:
import os, sys, json, logging, random
from dataclasses import dataclass, field
from pathlib import Path
from typing import Optional, Union


def get_logger(name: str = "ecommerce_llm", level: int = logging.INFO) -> logging.Logger:
    logger = logging.getLogger(name)
    logger.setLevel(level)
    if not logger.handlers:
        handler = logging.StreamHandler(sys.stdout)
        formatter = logging.Formatter(
            "%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
            datefmt="%H:%M:%S",
        )
        handler.setFormatter(formatter)
        logger.addHandler(handler)
    return logger


logger = get_logger("notebook07")


def set_seed(seed: int = 42) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np
        import torch
        np.random.seed(seed)
        torch.manual_seed(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
    except Exception:
        pass


@dataclass
class Config:
    project_root: Path = field(
        default_factory=lambda: Path(os.environ.get("PROJECT_ROOT", Path.cwd().parent)).resolve()
    )
    seed: int = 42
    base_model_name: str = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
    max_seq_length: int = 2048
    load_in_4bit: bool = True
    system_prompt: str = (
        "You are a helpful, polite Ecommerce Customer Support Assistant. "
        "You answer questions about orders, shipping, refunds, returns, "
        "payments, coupons, delivery, and account management. "
        "If a question is unrelated to ecommerce customer support, politely "
        "say that you can only help with ecommerce support questions."
    )
    instruction_header: str = "### Instruction:"
    response_header: str = "### Response:"
    default_temperature: float = 0.7
    default_max_new_tokens: int = 256
    default_top_p: float = 0.9
    lora_adapter_dir: Path = field(default_factory=lambda: Path(os.environ.get("PROJECT_ROOT", Path.cwd().parent)).resolve() / "models" / "lora_adapter")
    merged_model_dir: Path = field(default_factory=lambda: Path(os.environ.get("PROJECT_ROOT", Path.cwd().parent)).resolve() / "models" / "merged_model")

    def __post_init__(self) -> None:
        set_seed(self.seed)


class PromptFormatter:
    def __init__(self, config: Optional[Config] = None):
        self.config = config or Config()

    def format_training_example(self, instruction: str, response: str) -> str:
        cfg = self.config
        return (
            f"{cfg.instruction_header}\n"
            f"Customer:\n{instruction.strip()}\n\n"
            f"{cfg.response_header}\n"
            f"{response.strip()}"
        )

    def format_prompt_only(self, instruction: str) -> str:
        cfg = self.config
        return f"{cfg.instruction_header}\nCustomer:\n{instruction.strip()}\n\n{cfg.response_header}\n"

    def format_chat_messages(self, instruction: str, response: Optional[str] = None) -> list[dict]:
        messages = [
            {"role": "system", "content": self.config.system_prompt},
            {"role": "user", "content": instruction.strip()},
        ]
        if response is not None:
            messages.append({"role": "assistant", "content": response.strip()})
        return messages


def detect_device() -> str:
    try:
        import torch
        if torch.cuda.is_available():
            name = torch.cuda.get_device_name(0)
            logger.info(f"CUDA GPU detected: {name}")
            return "cuda"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
            logger.info("Apple MPS backend detected.")
            return "mps"
    except ImportError:
        logger.warning("PyTorch not installed yet; defaulting device check to CPU.")
    logger.warning("No GPU detected — falling back to CPU.")
    return "cpu"


class EcommerceSupportBot:
    def __init__(self, config: Optional[Config] = None):
        self.config = config or Config()
        self.formatter = PromptFormatter(self.config)
        self.model = None
        self.tokenizer = None
        self._loaded_from: Optional[str] = None

    def load_merged_model(self, model_dir: Optional[Union[str, Path]] = None):
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch

        model_dir = str(model_dir or self.config.merged_model_dir)
        logger.info(f"Loading merged model from {model_dir}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_dir,
            device_map="auto",
            torch_dtype=torch.bfloat16 if detect_device() == "cuda" else torch.float32,
        )
        self._loaded_from = "merged"
        return self.model, self.tokenizer

    def load_adapter_model(
        self,
        base_model_name: Optional[str] = None,
        adapter_dir: Optional[Union[str, Path]] = None,
    ):
        from unsloth import FastLanguageModel

        cfg = self.config
        base_model_name = base_model_name or cfg.base_model_name
        adapter_dir = str(adapter_dir or cfg.lora_adapter_dir)

        logger.info(f"Loading base model '{base_model_name}' + adapter '{adapter_dir}'")
        model, tokenizer = FastLanguageModel.from_pretrained(
            model_name=base_model_name,
            max_seq_length=cfg.max_seq_length,
            load_in_4bit=cfg.load_in_4bit,
        )
        model.load_adapter(adapter_dir)
        FastLanguageModel.for_inference(model)

        self.model, self.tokenizer = model, tokenizer
        self._loaded_from = "adapter"
        return model, tokenizer

    def merge_and_save(self, save_dir: Optional[Union[str, Path]] = None) -> Path:
        if self.model is None:
            raise RuntimeError("Load a base+adapter model first via load_adapter_model().")
        save_dir = Path(save_dir or self.config.merged_model_dir)
        save_dir.mkdir(parents=True, exist_ok=True)
        logger.info(f"Merging LoRA weights and saving to {save_dir} ...")
        merged = self.model.merge_and_unload() if hasattr(self.model, "merge_and_unload") else self.model
        merged.save_pretrained(str(save_dir))
        self.tokenizer.save_pretrained(str(save_dir))
        logger.info("Merge complete.")
        return save_dir

    def generate(
        self,
        instruction: str,
        max_new_tokens: Optional[int] = None,
        temperature: Optional[float] = None,
        top_p: Optional[float] = None,
        use_chat_template: bool = True,
    ) -> str:
        import torch

        if self.model is None or self.tokenizer is None:
            raise RuntimeError("No model loaded. Call load_merged_model() or load_adapter_model().")

        cfg = self.config
        max_new_tokens = max_new_tokens or cfg.default_max_new_tokens
        temperature = temperature if temperature is not None else cfg.default_temperature
        top_p = top_p if top_p is not None else cfg.default_top_p

        if use_chat_template and getattr(self.tokenizer, "chat_template", None):
            messages = self.formatter.format_chat_messages(instruction)
            input_ids = self.tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, return_tensors="pt"
            ).to(self.model.device)
            attention_mask = torch.ones_like(input_ids)
        else:
            prompt = self.formatter.format_prompt_only(instruction)
            enc = self.tokenizer(prompt, return_tensors="pt").to(self.model.device)
            input_ids, attention_mask = enc["input_ids"], enc["attention_mask"]

        with torch.no_grad():
            output_ids = self.model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                temperature=max(temperature, 1e-3),
                top_p=top_p,
                do_sample=temperature > 0,
                pad_token_id=self.tokenizer.eos_token_id,
            )

        new_tokens = output_ids[0][input_ids.shape[-1] :]
        response = self.tokenizer.decode(new_tokens, skip_special_tokens=True)
        return response.strip

    def chat(self, history: list[tuple[str, str]], new_message: str, **gen_kwargs) -> str:
        return self.generate(new_message, **gen_kwargs)


cfg = Config()


## 1. Load base model + LoRA adapter

In [ ]:
bot = EcommerceSupportBot(cfg)
model, tokenizer = bot.load_adapter_model()
print("Loaded base model + adapter for inference.")


## 2. Generate answers for a few sample questions

In [ ]:
sample_questions = [
    "Where is my order #48213?",
    "How do I return a pair of shoes that don't fit?",
    "My coupon code SAVE20 isn't applying at checkout.",
    "Can you help me book a flight to Paris?",  # off-topic, should be politely declined
]

for q in sample_questions:
    answer = bot.generate(q, temperature=0.6, max_new_tokens=150, use_chat_template=False)
    print("Q:", q)
    print("A:", answer)
    print("-" * 80)


## 3. Interactive chatbot loop

Run the cell below and type questions. Type `exit` or `quit` to stop.
(In Colab, this uses `input()` — works in the standard notebook UI.)

In [ ]:
def interactive_chat(bot, max_turns=20):
    print("Ecommerce Support Assistant — type 'exit' to stop.\n")
    for _ in range(max_turns):
        user_msg = input("You: ")
        if user_msg.strip().lower() in {"exit", "quit"}:
            print("Assistant: Thanks for chatting — have a great day!")
            break
        reply = bot.generate(user_msg, temperature=0.7, max_new_tokens=200, use_chat_template=False)
        print(f"Assistant: {reply}\n")

# Uncomment to run interactively:
# interactive_chat(bot)


## 4. Merge LoRA adapter into the base model

Merging produces a standalone model that can be loaded with plain `transformers` (no PEFT/Unsloth dependency needed at serve time) — this is what the Streamlit app loads by default.

In [ ]:
merged_dir = bot.merge_and_save(cfg.merged_model_dir)
print(f"Merged model saved to: {merged_dir}")
!ls -lh {merged_dir}


## 5. Verify the merged model loads and generates correctly

In [ ]:
verify_bot = EcommerceSupportBot(cfg)
verify_bot.load_merged_model(cfg.merged_model_dir)

test_answer = verify_bot.generate(
    "I never received a confirmation email after placing my order, what should I do?",
    temperature=0.6,
 )
print(test_answer)


## 6. (Optional) Push merged model to the Hugging Face Hub

Useful for deploying the Streamlit app on Streamlit Community Cloud / Spaces without re-uploading large weights manually.

In [ ]:
PUSH_TO_HUB = False  # flip to True and set your repo id + token to enable
HF_REPO_ID = "your-username/ecommerce-support-llm"

if PUSH_TO_HUB:
    from huggingface_hub import login
    login()  # will prompt for an HF token (free account)

    verify_bot.model.push_to_hub(HF_REPO_ID)
    verify_bot.tokenizer.push_to_hub(HF_REPO_ID)
    print(f"Pushed merged model to https://huggingface.co/{HF_REPO_ID}")
else:
    print("Skipping Hub push (PUSH_TO_HUB=False).")


## Summary

- Loaded base model + LoRA adapter and ran sample + interactive inference
- Confirmed the off-topic guardrail behavior (system prompt) works as expected
- Merged the adapter into a standalone model saved at `models/merged_model/`
- Verified the merged model loads and generates correctly with plain `transformers`
- Optionally pushed the merged model to the Hugging Face Hub

**Next:** run the Streamlit app locally:
```bash
streamlit run app/streamlit_app.py
```